In [10]:
# Bước 1: Clone source
!git clone https://github.com/sangtran0897/OmniVoice.git
%cd OmniVoice
!pip install -e .

# Bước 2: Trỏ Python tới source folder
import sys
sys.path.insert(0, '/content/omnivoice')



Cloning into 'OmniVoice'...
remote: Enumerating objects: 331, done.
remote: Counting objects: 100% (247/247), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 331 (delta 162), reused 139 (delta 139), pack-reused 84 (from 1)
Receiving objects: 100% (331/331), 47.48 MiB | 14.07 MiB/s, done.
Resolving deltas: 100% (172/172), done.
/content/OmniVoice/OmniVoice
Obtaining file:///content/OmniVoice/OmniVoice
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for omnivoice (pyproject.toml) ... done
  Created wheel for omnivoice: filename=omnivoice-0.1.5-py3-none-any.whl size=10961 sha256=c76aa5e5f6332eafcf07d7d868c34595b2a5da2757c53856f6c07f89a521e62c
  Stored in directory: /tmp/pip-ephem-wheel-cache-e008o67t/wheels/8a/ed/68/949115071d2dd7e8e310873ae562329

In [ ]:
# Bước 3: Code y chang như cũ
from omnivoice import OmniVoice
import soundfile as sf
import torch
from IPython.display import Audio, display

# model = OmniVoice.from_pretrained(
#     "kjanh/KhanhTTS-OmniVoice",
#     device_map="cuda:0",
#     dtype=torch.float16,
#     load_asr=True,
# )
model = OmniVoice.from_pretrained(
    "ngoctham/KhanhTTS-OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
    load_asr=True,
)
# model = OmniVoice.from_pretrained(
#     "termanteus/omnivoice-vietnamese",
#     device_map="cuda:0",
#     dtype=torch.float16,
#     load_asr=True,
# )

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

In [12]:
from huggingface_hub import hf_hub_download

print("Downloading reference audio from Hugging Face...")

ref_audio_path = hf_hub_download(
    repo_id="sangtran0897/omnivoice-clone",
    filename="miennamchuan_5s.WAV",
    repo_type="dataset",   # quan trọng: repo này là dataset
    token=hf_token
)

print(f"Loaded: {ref_audio_path}")


Loaded: /root/.cache/huggingface/hub/datasets--sangtran0897--omnivoice-clone/snapshots/3821845ea1267bba67c1e01b8df99ce7de5ade21/miennamchuan_5s.WAV


In [13]:
!git pull

Already up to date.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, Audio, clear_output
import soundfile as sf
import re
import subprocess
from IPython.display import Audio, display

def speedup_audio(input_path, output_path, speed=1.08):
    subprocess.run([
        "ffmpeg",
        "-y",
        "-i", input_path,
        "-filter:a", f"atempo={speed}",
        "-vn",
        output_path
    ], check=True)


output = widgets.Output()

ref_text = "để có thể đắp chút ánh hào quang rẻ tiền lên một gia tộc vốn đang khao khát có được sự chú ý."

def count_vietnamese_units(text):
    return len(re.findall(r"[A-Za-zÀ-ỹĐđ0-9]+", text))

def estimate_pause_seconds(text):
    comma = len(re.findall(r"[,，]", text)) * 0.12
    semi = len(re.findall(r"[;:；：]", text)) * 0.18
    end = len(re.findall(r"[.!?。！？]", text)) * 0.28
    newline = text.count("\n") * 0.35
    return comma + semi + end + newline

def estimate_duration_from_ref(text, ref_audio_path, ref_text, speed=1.0):
    ref_audio, sr = sf.read(ref_audio_path)
    ref_duration = len(ref_audio) / sr

    ref_units = max(count_vietnamese_units(ref_text), 1)
    target_units = max(count_vietnamese_units(text), 1)

    sec_per_unit = ref_duration / ref_units

    # Chặn biên để tránh audio mẫu có khoảng lặng làm duration bị lệch
    sec_per_unit = min(max(sec_per_unit, 0.22), 0.42)

    duration = target_units * sec_per_unit
    duration += estimate_pause_seconds(text)
    duration /= speed

    return round(max(duration, 1.2), 2)

text_box = widgets.Textarea(
    value="",
    placeholder="Paste đoạn text vào đây...",
    description="Text:",
    layout=widgets.Layout(width="100%", height="300px")
)

def on_text_change(change):
    if change["name"] == "value":
        input_text = change["new"].strip()
        if not input_text:
            return

        with output:
            clear_output(wait=True)
            print("Đang generate...")

            duration = estimate_duration_from_ref(
                input_text,
                ref_audio_path,
                ref_text,
                speed=1.0
            )

            print(f"Estimated duration: {duration}s")

            audio = model.generate(
                text=input_text,
                ref_audio=ref_audio_path,
                ref_text=ref_text,
                duration=duration,
                language="vi"
            )

            sf.write("clone_out.wav", audio[0], 24000)

            speedup_audio("clone_out.wav", "clone_out_fast.wav", speed=1.1)

            display(Audio("clone_out_fast.wav"))
            print("Xong.")

text_box.observe(on_text_change, names="value")

display(text_box, output)

Textarea(value='', description='Text:', layout=Layout(height='300px', width='100%'), placeholder='Paste đoạn t…

Output()